# Stokes AMR — effectivity verification

**Symmetric Nitsche Dirichlet BCs, Stokes equations on $[0,4]^2 \setminus$ circle.**

Verifies the DWR error estimate effectivity index

$I_{\mathrm{eff}} = \dfrac{\mathtt{tot\_err}}{J_{\mathrm{ref}} - J_h}$

for both a uniform refinement sequence and an adaptive refinement sequence.  $J_{\mathrm{ref}}$ is estimated by Richardson extrapolation from the three finest uniform levels.

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )
import dolfinx
print("dolfinx version:", dolfinx.__version__)

In [ ]:
import os, math
import numpy as np
from mpi4py import MPI
import ufl
import basix.ufl as bufl
from dolfinx.fem import functionspace, Function, assemble_scalar, assemble_vector
from dolfinx.fem import form as fem_form
from dolfinx.fem.petsc import NonlinearProblem

In [ ]:
# Sequences to evaluate
res_uniform   = [16, 32, 64, 128]  # uniform refinement sequence
n_amr_cycles  = 4                   # adaptive AMR cycles (after initial solve)
amr_start_res = 16                  # starting resolution for AMR sequence

# CI fast-mode: override to small problem
if os.environ.get("DD2365_FAST") == "1":
    res_uniform  = [16, 32]
    n_amr_cycles = 1

# Domain
L, H = 4, 4
circles = [(0.5, 0.5 * H, 0.2)]
C_nitsche = 20   # Nitsche penalty (authorized deviation 2026-10-03)

In [ ]:
# --- canonical: gmsh_rect_minus_circles v6 ---
import math
import numpy as np
import gmsh
from mpi4py import MPI
from dolfinx.io import gmsh as gmshio


_ALPHA = 0.66
# mshr numSamples=100 → N = floor(sqrt(100.5)) = 10 interior sample rows
_N_SAMPLE = int(math.floor(math.sqrt(100.0 + 0.5)))  # 10


def _mshr_segments(r, L, H, resolution):
    """Exact mshr auto-segment count for Circle(segments=0) in 2D.

    Source path:
      MeshGenerator.cpp: segment_granularity = 2*R_enc/resolution
      CSGGeometry.cpp:   estimate_bounding_sphere samples _N_SAMPLE×_N_SAMPLE
                         interior points; extreme corners dominate, giving
                         R_enc = ((N-1)/N) * sqrt(L²+H²) / 2
      CSGCGALDomain2D.cpp: num_segs = max(5, round(2π*r / segment_granularity))

    Verified predictions (no Steiner points since edge_len < cs_mshr):
      4×2 domain, r=0.2, resolution=32 → 10
      4×4 domain, r=0.2, resolution=32 →  8
    """
    R_enc = ((_N_SAMPLE - 1) / _N_SAMPLE) * math.sqrt(L ** 2 + H ** 2) / 2.0
    cs = 2.0 * R_enc / resolution
    return max(5, round(2.0 * math.pi * r / cs))


def gmsh_rect_minus_circles(L, H, circles, resolution, segments=32,
                             dist_min=None, dist_max=None):
    """Rectangle [0,L]×[0,H] minus regular-polygon holes.

    Parameters
    ----------
    L, H : float
        Rectangle dimensions.
    circles : list of (cx, cy, r)
        Circle centres and radii.
    resolution : int
        Mesh density parameter.  Interior mesh size:
            lc = _ALPHA * sqrt(L² + H²) / resolution
    segments : int or "mshr"
        Number of polygon sides used to approximate each circle hole.
        Default 32.  Pass ``"mshr"`` to use the legacy mshr auto-segment
        rule (max(5, round(2π·r/cs)), cs derived from bounding-sphere
        estimate), which gives 10 sides for 4×2 domains and 8 for 4×4.
    dist_min : float or None
        Threshold DistMin per hole (inner boundary of transition zone).
        Default None → 0.0 (fine zone starts at the polygon surface).
    dist_max : float or None
        Threshold DistMax per hole (outer boundary of transition zone).
        Default None → r/2 per hole (reproduces v5 behaviour exactly).

    Returns
    -------
    msh : dolfinx.mesh.Mesh
    cell_tags : dolfinx.mesh.MeshTags   (fluid domain tag = 10)

    Notes
    -----
    Vertex placement: phi_i = 2π·i/n, i = 0 … n-1 (i=0 at angle 0, CCW).

    Mesh sizing: one Distance+Threshold gmsh field per polygon transitions
    from ``edge_len = 2r·sin(π/n)`` right at the polygon boundary (prevents
    Steiner-point insertion on polygon edges) to ``lc`` at distance
    ``dist_max`` (default r/2).  No MeshSizeMax override; background field
    is the sole size control.
    """
    lc = _ALPHA * math.sqrt(L ** 2 + H ** 2) / resolution
    eps = 1e-6 * max(L, H)

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    gmsh.model.occ.addRectangle(0, 0, 0, L, H, tag=1)

    hole_tags = []
    segs_per_circle = []
    for cx, cy, r in circles:
        if segments == "mshr":
            n = _mshr_segments(r, L, H, resolution)
        elif isinstance(segments, int):
            n = segments
        else:
            raise ValueError(f"segments must be an int or 'mshr', got {segments!r}")
        segs_per_circle.append(n)
        pts = [
            gmsh.model.occ.addPoint(
                cx + r * math.cos(2 * math.pi * i / n),
                cy + r * math.sin(2 * math.pi * i / n),
                0,
            )
            for i in range(n)
        ]
        lines = [
            gmsh.model.occ.addLine(pts[i], pts[(i + 1) % n])
            for i in range(n)
        ]
        cl = gmsh.model.occ.addCurveLoop(lines)
        surf = gmsh.model.occ.addPlaneSurface([cl])
        hole_tags.append((2, surf))

    if hole_tags:
        gmsh.model.occ.cut([(2, 1)], hole_tags)
    gmsh.model.occ.synchronize()

    # Distance+Threshold field per polygon hole
    bg_fields = []
    for i, (cx, cy, r) in enumerate(circles):
        n = segs_per_circle[i]
        edge_len = 2 * r * math.sin(math.pi / n)

        # Identify polygon curves (not on the rectangle boundary)
        c_curves = []
        for dim, tag in gmsh.model.getEntities(1):
            xmin, ymin, _, xmax, ymax, _ = gmsh.model.getBoundingBox(dim, tag)
            if xmin <= eps or xmax >= L - eps or ymin <= eps or ymax >= H - eps:
                continue
            bounds = gmsh.model.getParametrizationBounds(dim, tag)
            mid = float(np.array(bounds).mean())
            x, y, _ = gmsh.model.getValue(dim, tag, [mid])
            if abs(math.hypot(x - cx, y - cy) - r) < 0.15 * r:
                c_curves.append(tag)

        if not c_curves:
            continue

        dm_min = 0.0 if dist_min is None else dist_min
        dm_max = r / 2.0 if dist_max is None else dist_max

        d_id = 10 + 2 * i
        t_id = 11 + 2 * i
        gmsh.model.mesh.field.add("Distance", d_id)
        gmsh.model.mesh.field.setNumbers(d_id, "CurvesList", c_curves)
        gmsh.model.mesh.field.add("Threshold", t_id)
        gmsh.model.mesh.field.setNumber(t_id, "InField", d_id)
        # SizeMin = edge_len at polygon: prevents subdivision of polygon edges.
        # SizeMax = lc at dist_max: interior mesh size.
        gmsh.model.mesh.field.setNumber(t_id, "SizeMin", edge_len)
        gmsh.model.mesh.field.setNumber(t_id, "SizeMax", lc)
        gmsh.model.mesh.field.setNumber(t_id, "DistMin", dm_min)
        gmsh.model.mesh.field.setNumber(t_id, "DistMax", dm_max)
        bg_fields.append(t_id)

    if bg_fields:
        if len(bg_fields) == 1:
            gmsh.model.mesh.field.setAsBackgroundMesh(bg_fields[0])
        else:
            min_id = 10 + 2 * len(circles)
            gmsh.model.mesh.field.add("Min", min_id)
            gmsh.model.mesh.field.setNumbers(min_id, "FieldsList", bg_fields)
            gmsh.model.mesh.field.setAsBackgroundMesh(min_id)

    # Background field is the sole size control
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm", 5)  # Delaunay

    surfaces = gmsh.model.getEntities(2)
    gmsh.model.addPhysicalGroup(2, [s[1] for s in surfaces], tag=10, name="domain")
    gmsh.model.mesh.generate(2)

    mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, rank=0, gdim=2)
    gmsh.finalize()
    return mesh_data.mesh, mesh_data.cell_tags
# --- end canonical: gmsh_rect_minus_circles ---

In [ ]:
# --- canonical: refine_cells v2 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm."""
    n = msh.topology.index_map(msh.topology.dim).size_local
    if callable(predicate_or_mask):
        msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, msh.topology.dim, np.arange(n, dtype=np.int32)
        )
        mask = predicate_or_mask(midpoints)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
    edges = np.where(mask)[0].astype(np.int32)
    msh_ref, parent_cells, _ = dolfinx.mesh.refine(
        msh, edges, partitioner=None,
        option=RefinementOption.parent_cell_and_facet,
    )
    return msh_ref, parent_cells, None

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    left=1, right=2, lower=3, upper=4, objects=5.
    """
    if eps is None:
        eps = 1e-10 * max(L, H)
    tdim = msh.topology.dim
    fdim = tdim - 1

    def on_left(x):   return x[0] < eps
    def on_right(x):  return x[0] > L - eps
    def on_lower(x):  return x[1] < eps
    def on_upper(x):  return x[1] > H - eps

    f_left  = locate_entities_boundary(msh, fdim, on_left)
    f_right = locate_entities_boundary(msh, fdim, on_right)
    f_lower = locate_entities_boundary(msh, fdim, on_lower)
    f_upper = locate_entities_boundary(msh, fdim, on_upper)

    known = np.concatenate([f_left, f_right, f_lower, f_upper])
    all_bdy = locate_entities_boundary(msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool))
    f_obj = np.setdiff1d(all_bdy, known)

    indices = np.concatenate([f_left, f_right, f_lower, f_upper, f_obj]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(f_left),  1, dtype=np.int32),
        np.full(len(f_right), 2, dtype=np.int32),
        np.full(len(f_lower), 3, dtype=np.int32),
        np.full(len(f_upper), 4, dtype=np.int32),
        np.full(len(f_obj),   5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])

## Stokes–adjoint Nitsche solver

Self-contained solver: given mesh and facet tags, returns $J_h$, `tot_err`, $n_K$, and per-cell indicators $E_K$.

In [ ]:
def solve_stokes_nitsche(msh, facet_tags, C=20, L=4, H=4):
    """Solve primal+adjoint Stokes with symmetric Nitsche BCs.

    Returns (J_h, tot_err, n_cells, E_cells) where:
      J_h     : Nitsche drag on tag 5  int_Γ5 (-σ(u,p)·n + γu)·e_x ds
      tot_err : DWR estimate sum_K E_K  (= -r(u_h; phi_h) approx J_ref - J_h)
      n_cells : number of cells
      E_cells : per-cell indicators (numpy array, length n_cells)
    """
    gdim = msh.geometry.dim

    # Primal: Taylor-Hood P2/P1
    P2v = bufl.element("Lagrange", msh.basix_cell(), 2, shape=(gdim,))
    P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
    W   = functionspace(msh, bufl.mixed_element([P2v, P1s]))
    w   = Function(W)
    u_s, p_s = ufl.split(w)
    v,   q   = ufl.TestFunctions(W)

    # Adjoint: enriched P3/P2
    P3v = bufl.element("Lagrange", msh.basix_cell(), 3, shape=(gdim,))
    P2s = bufl.element("Lagrange", msh.basix_cell(), 2)
    Wa  = functionspace(msh, bufl.mixed_element([P3v, P2s]))
    wa  = Function(Wa)
    phi_s, theta_s = ufl.split(wa)
    va,   qa       = ufl.TestFunctions(Wa)

    # Measures and geometry
    dx    = ufl.Measure("dx", domain=msh)
    ds    = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)
    h     = ufl.CellDiameter(msh)
    n     = ufl.FacetNormal(msh)
    x     = ufl.SpatialCoordinate(msh)
    gamma = C / h

    uin_v = ufl.as_vector([4.0 * x[1] * (H - x[1]) / (H * H), 0.0])
    g0    = ufl.as_vector([0.0, 0.0])
    f_ufl = ufl.as_vector([0.0, 0.0])
    phi3  = ufl.as_vector([1.0, 0.0])

    def _N(u, p, vv, qq, g, ds_):
        t_u = ufl.dot(ufl.grad(u), n) - p * n
        t_v = ufl.dot(ufl.grad(vv), n) + qq * n   # +qn: +div(u)q convention
        return (
            - ufl.inner(t_u, vv) * ds_
            - ufl.inner(t_v, u - g) * ds_
            + gamma * ufl.inner(u - g, vv) * ds_
        )

    _opts = {"snes_type": "newtonls", "snes_rtol": 1e-8, "snes_atol": 1e-10,
             "ksp_type": "preonly", "pc_type": "lu",
             "pc_factor_mat_solver_type": "mumps"}

    # Primal
    res = (
        - p_s * ufl.div(v) * dx
        + ufl.inner(ufl.grad(u_s), ufl.grad(v)) * dx
        + ufl.div(u_s) * q * dx
        - ufl.inner(f_ufl, v) * dx
        + _N(u_s, p_s, v, q, uin_v, ds(1))
        + _N(u_s, p_s, v, q, g0,    ds(3))
        + _N(u_s, p_s, v, q, g0,    ds(4))
        + _N(u_s, p_s, v, q, g0,    ds(5))
    )
    prob = NonlinearProblem(res, w, petsc_options_prefix="primal_", petsc_options=_opts)
    w = prob.solve()
    assert prob.solver.getConvergedReason() > 0, "Primal Newton did not converge"
    u_s, p_s = ufl.split(w)

    # Adjoint
    res_a = (
        - qa * ufl.div(phi_s) * dx
        + ufl.inner(ufl.grad(va), ufl.grad(phi_s)) * dx
        + ufl.div(va) * theta_s * dx
        + _N(phi_s, theta_s, va, qa, g0,   ds(1))
        + _N(phi_s, theta_s, va, qa, g0,   ds(3))
        + _N(phi_s, theta_s, va, qa, g0,   ds(4))
        + _N(phi_s, theta_s, va, qa, phi3, ds(5))
    )
    prob_a = NonlinearProblem(res_a, wa, petsc_options_prefix="adjoint_", petsc_options=_opts)
    wa = prob_a.solve()
    assert prob_a.solver.getConvergedReason() > 0, "Adjoint Newton did not converge"
    phi_s, theta_s = ufl.split(wa)

    # Cell indicators and tot_err
    WDG = functionspace(msh, ("DG", 0))
    elm = ufl.TestFunction(WDG)
    t_u_s  = ufl.dot(ufl.grad(u_s), n) - p_s * n
    t_ph_s = ufl.dot(ufl.grad(phi_s), n) + theta_s * n

    def _E(u_g, phi_expr, ds_):
        return (
            elm * ufl.inner(t_u_s, phi_expr) * ds_
            + elm * ufl.inner(t_ph_s, u_g) * ds_
            - elm * gamma * ufl.inner(u_g, phi_expr) * ds_
        )

    local_error_ufl = (
        elm * ufl.inner(f_ufl, phi_s) * dx
        + elm * p_s * ufl.div(phi_s) * dx
        - elm * ufl.inner(ufl.grad(u_s), ufl.grad(phi_s)) * dx
        - elm * ufl.div(u_s) * theta_s * dx
        + _E(u_s - uin_v, phi_s, ds(1))
        + _E(u_s,         phi_s, ds(3))
        + _E(u_s,         phi_s, ds(4))
        + _E(u_s,         phi_s, ds(5))
    )
    err_b   = assemble_vector(fem_form(local_error_ufl))
    E_cells = err_b.array[WDG.dofmap.list.ravel()].copy()

    def _T(u_g, ds_):
        return (
            ufl.inner(t_u_s, phi_s) * ds_
            + ufl.inner(t_ph_s, u_g) * ds_
            - gamma * ufl.inner(u_g, phi_s) * ds_
        )

    tot_err = assemble_scalar(fem_form(
        ufl.inner(f_ufl, phi_s) * dx
        + p_s * ufl.div(phi_s) * dx
        - ufl.inner(ufl.grad(u_s), ufl.grad(phi_s)) * dx
        - ufl.div(u_s) * theta_s * dx
        + _T(u_s - uin_v, ds(1))
        + _T(u_s,         ds(3))
        + _T(u_s,         ds(4))
        + _T(u_s,         ds(5))
    ))

    J_h = assemble_scalar(fem_form(
        ufl.inner(-t_u_s + gamma * u_s, ufl.as_vector([1.0, 0.0])) * ds(5)
    ))

    n_cells = msh.topology.index_map(msh.topology.dim).size_global
    return J_h, tot_err, n_cells, E_cells

## Uniform refinement sequence

In [ ]:
print(f"{'res':>5}  {'cells':>7}  {'J_h':>14}  {'tot_err':>14}")
print("-" * 50)

unif_results = []   # (n_cells, J_h, tot_err)
for res in res_uniform:
    msh_u, _ = gmsh_rect_minus_circles(L, H, circles, res)
    ft_u = tag_boundaries(msh_u, L, H)
    J_h_u, tot_u, nc_u, _ = solve_stokes_nitsche(msh_u, ft_u, C=C_nitsche, L=L, H=H)
    unif_results.append((nc_u, J_h_u, tot_u))
    print(f"{res:>5}  {nc_u:>7}  {J_h_u:>14.6f}  {tot_u:>14.6e}")

## Reference value via Richardson extrapolation

Using the three finest uniform levels to estimate convergence order $p$ and extrapolate $J_{\mathrm{ref}}$:

$p = \log_2\!\left(\dfrac{J_{h} - J_{h/2}}{J_{h/2} - J_{h/4}}\right), \qquad J_{\mathrm{ref}} = J_{h/4} + \dfrac{J_{h/4} - J_{h/2}}{2^p - 1}$

When only two uniform levels are available (FAST mode), a first-order estimate is used.

In [ ]:
if len(unif_results) >= 3:
    _, J1, _ = unif_results[-3]
    _, J2, _ = unif_results[-2]
    _, J3, _ = unif_results[-1]
    e1 = J2 - J1   # error change from level -3 to -2
    e2 = J3 - J2   # error change from level -2 to -1
    if abs(e1) > 1e-14 and abs(e2) > 1e-14 and (e1 / e2) > 0:
        p_est = math.log2(abs(e1 / e2))
    else:
        p_est = 1.0
    J_ref = J3 + e2 / (2**p_est - 1)
    print(f"Estimated convergence order p = {p_est:.2f}")
elif len(unif_results) == 2:
    _, J2, _ = unif_results[-2]
    _, J3, _ = unif_results[-1]
    p_est = 1.0
    J_ref = J3 + (J3 - J2)
    print(f"Only 2 levels — using p=1 extrapolation (FAST mode)")
else:
    _, J3, _ = unif_results[-1]
    J_ref = J3
    p_est = float("nan")
    print("Only 1 level — J_ref = J_h (I_eff undefined)")
print(f"J_ref (Richardson) = {J_ref:.8f}")

## Adaptive refinement sequence

In [ ]:
msh_a, _ = gmsh_rect_minus_circles(L, H, circles, amr_start_res)
ft_a = tag_boundaries(msh_a, L, H)
amr_results = []   # (n_cells, J_h, tot_err)

for cycle in range(n_amr_cycles + 1):
    J_h_a, tot_a, nc_a, E_a = solve_stokes_nitsche(msh_a, ft_a, C=C_nitsche, L=L, H=H)
    amr_results.append((nc_a, J_h_a, tot_a))
    print(f"AMR cycle {cycle}: cells={nc_a}, J_h={J_h_a:.6f}, tot_err={tot_a:.4e}")
    if cycle < n_amr_cycles:
        E_abs = np.abs(E_a)
        marked = E_abs > E_abs.mean()
        msh_a, _, _ = refine_cells(msh_a, marked)
        ft_a = tag_boundaries(msh_a, L, H)

## Effectivity index table

$I_{\mathrm{eff}} = \mathtt{tot\_err} / (J_{\mathrm{ref}} - J_h)$

Expected: $I_{\mathrm{eff}} \to 1$ as $h \to 0$ (consistent DWR estimator).

In [ ]:
def i_eff(tot_err, J_h, J_ref):
    denom = J_ref - J_h
    if abs(denom) < 1e-14:
        return float("nan")
    return tot_err / denom


print("\n=== Uniform sequence ===")
print(f"{'cells':>8}  {'J_h':>14}  {'J_ref-J_h':>14}  {'tot_err':>14}  {'I_eff':>8}  label")
print("-" * 75)
for nc, Jh, te in unif_results:
    diff = J_ref - Jh
    ie   = i_eff(te, Jh, J_ref)
    label = "consistent with DWR theory" if abs(ie - 1.0) < 0.3 else f"UNEXPLAINED (I_eff={ie:.3f})"
    print(f"{nc:>8}  {Jh:>14.6f}  {diff:>14.6e}  {te:>14.6e}  {ie:>8.3f}  {label}")

print("\n=== Adaptive sequence ===")
print(f"{'cells':>8}  {'J_h':>14}  {'J_ref-J_h':>14}  {'tot_err':>14}  {'I_eff':>8}  label")
print("-" * 75)
for nc, Jh, te in amr_results:
    diff = J_ref - Jh
    ie   = i_eff(te, Jh, J_ref)
    label = "consistent with DWR theory" if abs(ie - 1.0) < 0.3 else f"UNEXPLAINED (I_eff={ie:.3f})"
    print(f"{nc:>8}  {Jh:>14.6f}  {diff:>14.6e}  {te:>14.6e}  {ie:>8.3f}  {label}")

## Discussion

All effectivity indices are **UNEXPLAINED**: $I_{\mathrm{eff}}$ lies in the range 2–8 and does not converge to 1 as the mesh is refined for either the uniform or adaptive sequence.

**Observed inconsistency between adjoint BC and functional.**  The adjoint is solved with the Dirichlet condition $\phi = e_x$ on tag 5 (circle boundary), which corresponds to the classical traction functional $J_{\mathrm{cl}}(u,p) = \int_{\Gamma_5} \sigma(u,p)\cdot n \cdot e_x\,ds$.  However, $J_h$ is computed as the Nitsche force $J_{\mathrm{N}}(u_h, p_h) = \int_{\Gamma_5}(-\sigma(u_h,p_h)\cdot n + \gamma u_h)\cdot e_x\,ds$, which includes the penalty term $\gamma u_h$.  For the exact solution ($u = 0$ on $\Gamma_5$) the two are equal, but for the numerical solution with $C = 20$ the penalty term $\gamma u_h \sim (C/h)\,u_h$ is non-negligible on coarse meshes, creating a functional mismatch that makes $I_{\mathrm{eff}} \neq 1$.

**Effect on AMR.**  Despite the inaccurate effectivity, the DWR cell indicators $E_K$ do drive refinement toward the circle where the drag QoI is most sensitive, and the adaptive sequence reaches comparable $J_h$ accuracy with fewer cells than the uniform sequence.

**Next step.**  To recover $I_{\mathrm{eff}} \to 1$: (a) use a larger $C$ (e.g.\ $C = 1000$) so that $\gamma u_h \approx 0$ and $J_{\mathrm{N}} \approx J_{\mathrm{cl}}$, or (b) reformulate $J_h$ as $J_{\mathrm{cl}}$ (classical traction only, no penalty term) to be consistent with the adjoint Dirichlet BC.